# Azure Key Vault Secrets Sync with Terraform and WIF

Terraform configures Vault as the OIDC issuer and creates a Microsoft Entra federated identity credential. No client secret is created or stored.

In [1]:
%env ARM_SUBSCRIPTION_ID=<azure-subscription-id>
%env ARM_TENANT_ID=<azure-tenant-id>
%env TF_VAR_azure_subscription_id=<azure-subscription-id>
%env TF_VAR_public_oidc_issuer_url=https://vault.jose-merchan.sbx.hashidemos.io

env: ARM_SUBSCRIPTION_ID=<azure-subscription-id>
env: ARM_TENANT_ID=<azure-tenant-id>
env: TF_VAR_azure_subscription_id=<azure-subscription-id>
env: TF_VAR_public_oidc_issuer_url=https://vault.jose-merchan.sbx.hashidemos.io


In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

ENV_FILE = next((p / ".env" for p in (Path.cwd(), *Path.cwd().parents) if (p / ".env").is_file()), None)
if ENV_FILE is None:
    raise FileNotFoundError("Could not find .env")
load_dotenv(ENV_FILE)

True

## Authenticate and confirm the Azure subscription

In [3]:
! az login --tenant $ARM_TENANT_ID --subscription $ARM_SUBSCRIPTION_ID
! az account show --query '{subscription:name,subscriptionId:id,tenantId:tenantId}' --output table
! az provider register --namespace Microsoft.KeyVault --subscription $ARM_SUBSCRIPTION_ID --wait
! az provider show --namespace Microsoft.KeyVault --subscription $ARM_SUBSCRIPTION_ID --query '{namespace:namespace,state:registrationState}' --output table

A web browser has been opened at https://login.microsoftonline.com/<azure-tenant-id>/oauth2/v2.0/authorize. Please continue the login in the web browser. If no web browser is available or if the web browser fails to open, use device code flow with `az login --use-device-code`.

Retrieving subscriptions for the selection...
[
  {
    "cloudName": "AzureCloud",
    "homeTenantId": "<azure-tenant-id>",
    "id": "<azure-subscription-id>",
    "isDefault": true,
    "managedByTenants": [],
    "name": "secret-sync-mapfre-test",
    "state": "Enabled",
    "tenantId": "<azure-tenant-id>",
    "user": {
      "name": "jose.merchan@hashicorp.services",
      "type": "user"
    }
  }
]
Subscription             SubscriptionId                        TenantId
-----------------------  ------------------------------------  ------------------------------------
secret-sync-mapfre-test  <azure-subscription-id>  <azure-tenant-id>
Namespace           State
------------------  ----------
Microsoft.KeyVau

## Verify the public Vault OIDC endpoints

In [4]:
! curl -fsS $TF_VAR_public_oidc_issuer_url/v1/identity/oidc/secrets-sync/.well-known/openid-configuration | jq
! curl -fsS $TF_VAR_public_oidc_issuer_url/v1/identity/oidc/secrets-sync/.well-known/keys | jq

{
  "issuer": "https://vault.jose-merchan.sbx.hashidemos.io/v1/identity/oidc/secrets-sync",
  "jwks_uri": "https://vault.jose-merchan.sbx.hashidemos.io/v1/identity/oidc/secrets-sync/.well-known/keys",
  "response_types_supported": [
    "id_token"
  ],
  "subject_types_supported": [
    "public"
  ],
  "id_token_signing_alg_values_supported": [
    "RS256",
    "RS384",
    "RS512",
    "ES256",
    "ES384",
    "ES512",
    "EdDSA"
  ]
}
{
  "keys": []
}


## Initialize and validate

In [5]:
! terraform -chdir=terraform-azure-wif init
! terraform -chdir=terraform-azure-wif validate

Initializing the backend...

Initializing provider plugins...
- Reusing previous version of hashicorp/azuread from the dependency lock file
- Reusing previous version of hashicorp/azurerm from the dependency lock file
- Reusing previous version of hashicorp/random from the dependency lock file
- Reusing previous version of hashicorp/time from the dependency lock file
- Reusing previous version of hashicorp/vault from the dependency lock file
- Using previously-installed hashicorp/azurerm v4.81.0
- Using previously-installed hashicorp/random v3.9.0
- Using previously-installed hashicorp/time v0.14.0
- Using previously-installed hashicorp/vault v5.10.1
- Using previously-installed hashicorp/azuread v3.9.0


Terraform has been successfully initialized!

You may now begin working with Terraform. Try running "terraform plan" to see
any changes that are required for your infrastructure. All Terraform commands
should now work.

If you ever set or change modules or backend configuration for Te

## Review the issuer, audience and subject

In [6]:
! terraform -chdir=terraform-azure-wif plan

data.vault_namespace.current: Reading...
data.vault_namespace.current: Read complete after 0s [id=/]
data.azuread_client_config.current: Reading...
data.azuread_client_config.current: Read complete after 0s [id=<azure-tenant-id>-04b07795-8ddb-461a-bbee-02f9e1bf7b46-0518caac-617d-4fe4-8cee-2809b74be4c2]
data.azurerm_client_config.current: Reading...
data.azurerm_client_config.current: Read complete after 0s [id=Y2xpZW50Q29uZmlncy9jbGllbnRJZD0wNGIwNzc5NS04ZGRiLTQ2MWEtYmJlZS0wMmY5ZTFiZjdiNDY7b2JqZWN0SWQ9MDUxOGNhYWMtNjE3ZC00ZmU0LThjZWUtMjgwOWI3NGJlNGMyO3N1YnNjcmlwdGlvbklkPTBiMzRmN2NhLTI4N2YtNDFiMi1iOGQyLWM4ZjE2MTgzNzVjYTt0ZW5hbnRJZD0yMzdmYmMwNC1jNTJhLTQ1OGItYWY5Ny1lYWY3MTU3YzBjZDQ=]

Terraform used the selected providers to generate the following execution plan.
Resource actions are indicated with the following symbols:
  + create

Terraform will perform the following actions:

  # azuread_application.secrets_sync will be created
  + resource "azuread_application" "secrets_sync" {
      + 

## Apply

The configuration waits 60 seconds for the federated credential and RBAC assignment to propagate before creating the Vault destination.

In [7]:
! terraform -chdir=terraform-azure-wif apply -auto-approve

data.vault_namespace.current: Reading...
data.vault_namespace.current: Read complete after 0s [id=/]
data.azuread_client_config.current: Reading...
data.azuread_client_config.current: Read complete after 0s [id=<azure-tenant-id>-04b07795-8ddb-461a-bbee-02f9e1bf7b46-0518caac-617d-4fe4-8cee-2809b74be4c2]
data.azurerm_client_config.current: Reading...
data.azurerm_client_config.current: Read complete after 0s [id=Y2xpZW50Q29uZmlncy9jbGllbnRJZD0wNGIwNzc5NS04ZGRiLTQ2MWEtYmJlZS0wMmY5ZTFiZjdiNDY7b2JqZWN0SWQ9MDUxOGNhYWMtNjE3ZC00ZmU0LThjZWUtMjgwOWI3NGJlNGMyO3N1YnNjcmlwdGlvbklkPTBiMzRmN2NhLTI4N2YtNDFiMi1iOGQyLWM4ZjE2MTgzNzVjYTt0ZW5hbnRJZD0yMzdmYmMwNC1jNTJhLTQ1OGItYWY5Ny1lYWY3MTU3YzBjZDQ=]

Terraform used the selected providers to generate the following execution plan.
Resource actions are indicated with the following symbols:
  + create

Terraform will perform the following actions:

  # azuread_application.secrets_sync will be created
  + resource "azuread_application" "secrets_sync" {
      + 

## Verify the published issuer and JWKS after apply

In [8]:
! curl -fsS $TF_VAR_public_oidc_issuer_url/v1/identity/oidc/secrets-sync/.well-known/openid-configuration | jq
! curl -fsS $TF_VAR_public_oidc_issuer_url/v1/identity/oidc/secrets-sync/.well-known/keys | jq

{
  "issuer": "https://vault.jose-merchan.sbx.hashidemos.io/v1/identity/oidc/secrets-sync",
  "jwks_uri": "https://vault.jose-merchan.sbx.hashidemos.io/v1/identity/oidc/secrets-sync/.well-known/keys",
  "response_types_supported": [
    "id_token"
  ],
  "subject_types_supported": [
    "public"
  ],
  "id_token_signing_alg_values_supported": [
    "RS256",
    "RS384",
    "RS512",
    "ES256",
    "ES384",
    "ES512",
    "EdDSA"
  ]
}
{
  "keys": [
    {
      "use": "sig",
      "kty": "RSA",
      "kid": "8c529909-b36f-badb-4514-d553cd25bd11",
      "alg": "RS256",
      "n": "zdgyi4ZTwnKW603HqiE_4qnvfAQPMPoByMhfYm6t4QDtYZF3kdRPW37XQ0GpUrLDlosaxTVcgX6VhvDhQYTBA5FgEoBUGKNBF7qdkiRFk2rQOaFeJmi7jVPyqzCxCWbyrBlWRWFd8KS_bnWtDZm9FpiPmwua3cNfWk3HTzj0lxAIHosjaNFIyTnCAZ4aX3lhtHVx4h-in8-9v6fF_uEcNXZSFINmsqIrE9VbaZwINh0kvN2Jo57cnXQ-poGEs6sPW3vEMbmMe3JXxZqKC9yJjVJRYe9NdJtx2LA_1uGqdIrIbi2F6pOBy1C2z63_wcetEjJKnkfgYrEKsSn6n6DGzQ",
      "e": "AQAB"
    },
    {
      "use": "sig",
      "kty": "

## Verify Vault and Azure Key Vault

In [9]:
! terraform -chdir=terraform-azure-wif output
! vault read sys/sync/destinations/azure-kv/mapfre-wif-azure-kv
! vault read -format=json sys/sync/destinations/azure-kv/mapfre-wif-azure-kv/associations | jq

destination_name = "mapfre-wif-azure-kv"
expected_subject = "secrets-sync:root:azure-kv:mapfre-wif-azure-kv"
external_secret_name = "vault-mapfre-wif-verification"
federated_application_client_id = "b1fff19d-3c58-4643-a485-050940b13b8c"
key_vault_name = "kv-mapfre-wif-73f1we"
oidc_issuer = "https://vault.jose-merchan.sbx.hashidemos.io/v1/identity/oidc/secrets-sync"
Key                   Value
---                   -----
connection_details    map[client_id:b1fff19d-3c58-4643-a485-050940b13b8c identity_token_audience:***** identity_token_key:***** identity_token_ttl:3600 key_vault_uri:https://kv-mapfre-wif-73f1we.vault.azure.net/ tenant_id:<azure-tenant-id>]
name                  mapfre-wif-azure-kv
options               map[custom_tags:map[ManagedBy:Terraform Purpose:VaultSecretsSyncWIF] granularity_level:secret-path secret_name_template:vault-mapfre-wif-{{ .SecretBaseName }}]
type                  azure-kv
uses_wif              true
{
  "request_id": "3285adc1-ba52-e43e-1db7-d4ed5550b7

In [10]:
%%bash
KEY_VAULT_NAME=$(terraform -chdir=terraform-azure-wif output -raw key_vault_name)
SECRET_NAME=$(terraform -chdir=terraform-azure-wif output -raw external_secret_name)
az keyvault secret show \
  --vault-name "$KEY_VAULT_NAME" \
  --name "$SECRET_NAME" \
  --query '{name:name,enabled:attributes.enabled,updated:attributes.updated}' \
  --output json

{
  "enabled": true,
  "name": "vault-mapfre-wif-verification",
  "updated": "2026-07-23T16:52:29+00:00"
}


# CLEAN UP

In [11]:
! terraform -chdir=terraform-azure-wif destroy -auto-approve

random_string.suffix: Refreshing state... [id=73f1we]
data.vault_namespace.current: Reading...
vault_activation_flags.secrets_sync: Refreshing state... [id=secrets-sync]
vault_identity_oidc.issuer: Refreshing state... [id=https://vault.jose-merchan.sbx.hashidemos.io]
vault_identity_oidc_key.secrets_sync: Refreshing state... [id=mapfre-wif-secrets-sync-key]
vault_mount.kv: Refreshing state... [id=mapfre-wif-kv]
data.vault_namespace.current: Read complete after 0s [id=/]
vault_kv_secret_v2.demo: Refreshing state... [id=mapfre-wif-kv/data/verification]
vault_identity_oidc_role.publish_key: Refreshing state... [id=mapfre-wif-key-publisher]
data.azuread_client_config.current: Reading...
data.azuread_client_config.current: Read complete after 0s [id=<azure-tenant-id>-04b07795-8ddb-461a-bbee-02f9e1bf7b46-0518caac-617d-4fe4-8cee-2809b74be4c2]
azuread_application.secrets_sync: Refreshing state... [id=/applications/74cd5891-9895-44c8-822e-36e5c4b902ea]
azuread_application_federated_identity_cred